# Five drift-mitigation pathways, evaluated against the MLP baseline

Same forward-chaining protocol as `mlp_final.ipynb` (train on `S_1..S_{T-1}`, validate on `S_T`,
report mean and large-drift-fold macro-F1), so every number here is directly comparable to that
notebook's baseline.

| # | Pathway | What's implemented |
|---|---|---|
| 1 | Drift-robust feature engineering | per-batch Z-score standardisation (OSC skipped, see note below) |
| 2 | Subspace alignment | PCA on the pooled source+target features (TCA skipped, see note below) |
| 3 | Domain-adversarial NN (DANN) | shared trunk + class head + gradient-reversal domain head |
| 4 | Dynamic ensemble | one weak MLP per training batch, recency-weighted vote |
| 5 | Semi-supervised pseudo-labeling | confident predictions on the target batch fed back as training data |

**Two substitutions from the brief, both for the same reason -- O(n^3)/O(n) blowup on this dataset's scale:**
- **OSC -> per-batch Z-score.** Full NIPALS-style OSC needs `y` to define the unwanted direction and
  is iterative; on this dataset the unwanted direction is just the per-batch mean/variance shift,
  which per-batch standardisation removes directly for the cost of a `groupby`.
- **TCA -> PCA-align.** Kernel TCA solves an eigenproblem on an `n x n` matrix where `n` is
  source+target rows. By fold 9 that's ~10,300 rows -> a 10,300x10,300 eigendecomposition, which is
  not something to casually run in a notebook. PCA on the pooled features gets the same
  "shared low-dimensional subspace" property via SVD on the `n x d` matrix instead (`d=129`),
  which is what the paper's own Path 2 description names as the other valid option.

Each pathway is a function `(train_df, apply_df) -> probs`, so the same function evaluates every
CV fold *and* produces the final submission -- no separate "final" reimplementation.

Everything below only **builds and evaluates**. Nothing is executed here; run it yourself.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "  [no GPU]"))

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]      # 16 sensors x 8 descriptors, sensor-major
COLS = FEAT + ["concentration"]
N_SENSORS, N_DESC = 16, 8
CLASSES = sorted(train["gas_class"].unique())
K = len(CLASSES)
FOLDS = sorted(train["batch"].unique())[1:]
TEST_QUOTA = 600

_s = StandardScaler().fit(train[FEAT])
_X = _s.transform(train[FEAT])
_c = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT = {b: float(np.linalg.norm(_c[b] - _c[b - 1])) for b in FOLDS}
LARGE_DRIFT = [b for b, s in DRIFT.items() if s >= 5.0]

print(f"large-drift folds: {[int(b) for b in LARGE_DRIFT]}")


device: cuda  (NVIDIA GeForce RTX 5060 Ti)
large-drift folds: [2, 3, 4, 5, 6, 8]


## Shared baseline machinery

Copied from `mlp_final.ipynb` verbatim (signed-log prep, plain MLP, seed-averaged training,
macro-F1 helper, balanced-assignment rules) so the baseline row below is the same model, not a
reimplementation that might drift from it.

In [2]:
def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


def prep(fit_df, *apply_dfs):
    """signed-log + standardise, fit on the training fold only."""
    sc = StandardScaler().fit(signed_log(fit_df[COLS].values))
    out = [sc.transform(signed_log(d[COLS].values)).astype(np.float32)
           for d in (fit_df,) + apply_dfs]
    return out, sc.scale_[:len(FEAT)].astype(np.float32)


class MLP(nn.Module):
    def __init__(self, n_in, width=256, p_drop=0.3, n_out=K):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_out))

    def forward(self, x):
        return self.net(x)


def train_seeds(X_tr, y_tr, X_ap, feat_scale, n_seeds=5, aug_sigma=0.0,
                epochs=80, bs=256, lr=2e-3, wd=1e-4):
    """Returns probabilities averaged is left to the caller; shape (n_seeds, len(X_ap), K)."""
    Xt = torch.tensor(X_tr, device=DEVICE)
    Xa = torch.tensor(X_ap, device=DEVICE)
    yt = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    scale_t = torch.tensor(feat_scale, device=DEVICE)
    n_feat = len(feat_scale)
    probs = np.zeros((n_seeds, len(X_ap), K), dtype=np.float64)

    for s in range(n_seeds):
        torch.manual_seed(s)
        m = MLP(Xt.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for _ in range(epochs):
            m.train()
            for idx in torch.randperm(len(Xt), device=DEVICE).split(bs):
                if len(idx) < 2:
                    continue
                xb = Xt[idx]
                if aug_sigma > 0 and n_feat == len(FEAT):
                    d = torch.randn(len(idx), N_SENSORS, device=DEVICE) * aug_sigma
                    d = d.repeat_interleave(N_DESC, dim=1)
                    xb = xb.clone()
                    xb[:, :n_feat] += d / scale_t
                loss = F.cross_entropy(m(xb), yt[idx])
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
        m.eval()
        with torch.no_grad():
            probs[s] = F.softmax(m(Xa), dim=1).cpu().numpy()
    return probs


def f1_of(P, y):
    return f1_score(y, np.array(CLASSES)[P.argmax(1)], average="macro")


def sinkhorn(P, quota, iters=200, eps=1e-12):
    Q = np.clip(P, eps, None).copy()
    for _ in range(iters):
        Q /= Q.sum(1, keepdims=True)
        Q *= (quota / np.maximum(Q.sum(0), eps))
    return Q


def capped_greedy(P, quota):
    remaining = np.array(quota, dtype=int).copy()
    out = np.full(len(P), -1, dtype=int)
    for i in np.argsort(-P.max(1)):
        for c in np.argsort(-P[i]):
            if remaining[c] > 0:
                out[i] = c
                remaining[c] -= 1
                break
    return out


def baseline_fn(tr, ap, seeds=5, aug_sigma=0.0):
    """The MLP baseline. Every pathway below is compared against this, and several call it."""
    (Xt, Xa), fscale = prep(tr, ap)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    return train_seeds(Xt, y, Xa, fscale, n_seeds=seeds, aug_sigma=aug_sigma).mean(0)


## CV runner

One function walks `FOLDS` for any `(train_df, apply_df) -> probs` callable and returns per-fold
probabilities + labels, so every pathway (and every blend with the baseline) is scored the same
way and the results are directly comparable.

In [3]:
def eval_path(fn, seeds_note=""):
    probs, ys = {}, {}
    t0 = time.time()
    for vb in FOLDS:
        tr = train[train["batch"] < vb]
        va = train[train["batch"] == vb]
        probs[vb] = fn(tr, va)
        ys[vb] = va["gas_class"].values
    print(f"  [{time.time() - t0:.0f}s]" + (f"  {seeds_note}" if seeds_note else ""))
    return probs, ys


def summarize(name, probs, ys):
    per_fold = {vb: f1_of(probs[vb], ys[vb]) for vb in FOLDS}
    return dict(name=name, mean=np.mean(list(per_fold.values())),
                large_drift=np.mean([per_fold[b] for b in LARGE_DRIFT]))


results = []  # list of summary dicts, appended to as each cell below runs
ALL_PROBS = {}  # name -> {batch: probs}, kept for blending with the baseline

print("=== baseline: plain MLP, signed-log + standardise ===")
BASE_PROBS, BASE_Y = eval_path(lambda tr, ap: baseline_fn(tr, ap, seeds=5))
ALL_PROBS["baseline"] = BASE_PROBS
results.append(summarize("baseline", BASE_PROBS, BASE_Y))
pd.DataFrame(results)


=== baseline: plain MLP, signed-log + standardise ===
  [90s]


,name,mean,large_drift
0,baseline,0.849122,0.860706


## Path 1 -- drift-robust feature engineering (per-batch Z-score)

Each batch (train or target) is standardised against *its own* mean/std, computed after the same
`signed_log` transform. This removes the sensor-ageing mean/variance shift directly, batch by
batch, with no fitted parameters to transfer across the drift gap.

In [4]:
def prep_batchstd(fit_df, apply_df):
    def per_batch_z(df):
        X = signed_log(df[COLS].values).astype(np.float64)
        out = np.empty_like(X)
        for b in df["batch"].unique():
            m = (df["batch"].values == b)
            mu, sd = X[m].mean(0), X[m].std(0) + 1e-8
            out[m] = (X[m] - mu) / sd
        return out.astype(np.float32)
    return per_batch_z(fit_df), per_batch_z(apply_df)


def path1_batchstd_fn(tr, ap, seeds=5):
    Xt, Xa = prep_batchstd(tr, ap)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    fscale = np.ones(len(FEAT), dtype=np.float32)
    return train_seeds(Xt, y, Xa, fscale, n_seeds=seeds).mean(0)


print("=== path 1: per-batch Z-score standardisation ===")
P1_PROBS, P1_Y = eval_path(lambda tr, ap: path1_batchstd_fn(tr, ap, seeds=5))
ALL_PROBS["path1_batchstd"] = P1_PROBS
results.append(summarize("path1_batchstd", P1_PROBS, P1_Y))
pd.DataFrame(results)


=== path 1: per-batch Z-score standardisation ===
  [85s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path1_batchstd,0.795339,0.760929


## Path 2 -- subspace alignment (PCA-align)

PCA is fit on the pooled, unlabeled source+target features (no leakage: no labels used), then both
are projected into that shared low-dimensional subspace before the classifier ever sees them.

In [5]:
def prep_pca_align(fit_df, apply_df, n_components=40):
    (Xf, Xa), _ = prep(fit_df, apply_df)
    pca = PCA(n_components=n_components, random_state=0).fit(np.vstack([Xf, Xa]))
    return pca.transform(Xf).astype(np.float32), pca.transform(Xa).astype(np.float32)


def path2_pca_fn(tr, ap, seeds=5, n_components=40):
    Xt, Xa = prep_pca_align(tr, ap, n_components)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    fscale = np.ones(n_components, dtype=np.float32)
    return train_seeds(Xt, y, Xa, fscale, n_seeds=seeds).mean(0)


print("=== path 2: PCA subspace alignment ===")
P2_PROBS, P2_Y = eval_path(lambda tr, ap: path2_pca_fn(tr, ap, seeds=5))
ALL_PROBS["path2_pca"] = P2_PROBS
results.append(summarize("path2_pca", P2_PROBS, P2_Y))
pd.DataFrame(results)


=== path 2: PCA subspace alignment ===
  [85s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path1_batchstd,0.795339,0.760929
2,path2_pca,0.880155,0.898716


## Path 3 -- Domain-Adversarial Neural Network (DANN)

Same trunk shape as the baseline `MLP`. A gradient-reversal layer feeds the trunk's features to a
domain head that tries to tell source (training batches) from target (the batch being predicted);
reversing its gradient pushes the trunk to *stop* encoding batch identity. `lambda` ramps from 0 to
1 over training on the standard DANN schedule so the domain signal doesn't dominate early when the
trunk's features are still meaningless.

In [6]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd):
    return GradReverse.apply(x, lambd)


class DANN(nn.Module):
    def __init__(self, n_in, width=256, p_drop=0.3, n_out=K):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop))
        self.class_head = nn.Linear(128, n_out)
        self.domain_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, lambd=0.0):
        feat = self.trunk(x)
        return self.class_head(feat), self.domain_head(grad_reverse(feat, lambd))


def train_dann(tr_df, ap_df, seeds=5, epochs=80, bs=256, lr=2e-3, wd=1e-4):
    (Xt, Xa), _ = prep(tr_df, ap_df)
    y = np.searchsorted(CLASSES, tr_df["gas_class"].values)
    Xt_t = torch.tensor(Xt, device=DEVICE)
    Xa_t = torch.tensor(Xa, device=DEVICE)
    yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    n_steps_total = epochs * max(1, len(Xt_t) // bs)
    probs = np.zeros((seeds, len(Xa), K), dtype=np.float64)

    for s in range(seeds):
        torch.manual_seed(s)
        m = DANN(Xt_t.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        step = 0
        for _ in range(epochs):
            m.train()
            for idx in torch.randperm(len(Xt_t), device=DEVICE).split(bs):
                if len(idx) < 2:
                    continue
                p = step / n_steps_total
                lambd = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0
                tgt_idx = torch.randint(0, len(Xa_t), (len(idx),), device=DEVICE)
                xb_s, yb_s, xb_t = Xt_t[idx], yt[idx], Xa_t[tgt_idx]

                class_logits, dom_s = m(xb_s, lambd)
                _, dom_t = m(xb_t, lambd)
                class_loss = F.cross_entropy(class_logits, yb_s)
                dom_logits = torch.cat([dom_s, dom_t], 0).squeeze(-1)
                dom_labels = torch.cat([torch.zeros(len(idx)), torch.ones(len(idx))]).to(DEVICE)
                dom_loss = F.binary_cross_entropy_with_logits(dom_logits, dom_labels)
                loss = class_loss + dom_loss

                opt.zero_grad(); loss.backward(); opt.step()
                step += 1
            sch.step()
        m.eval()
        with torch.no_grad():
            probs[s] = F.softmax(m(Xa_t, 0.0)[0], dim=1).cpu().numpy()
    return probs.mean(0)


print("=== path 3: DANN (this one is slower -- gradient-reversal domain head) ===")
P3_PROBS, P3_Y = eval_path(lambda tr, ap: train_dann(tr, ap, seeds=5))
ALL_PROBS["path3_dann"] = P3_PROBS
results.append(summarize("path3_dann", P3_PROBS, P3_Y))
pd.DataFrame(results)


=== path 3: DANN (this one is slower -- gradient-reversal domain head) ===
  [197s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path1_batchstd,0.795339,0.760929
2,path2_pca,0.880155,0.898716
3,path3_dann,0.760262,0.717368


## Path 4 -- dynamic ensemble (recency-weighted)

One weak MLP per *individual* training batch (not cumulative), combined with weights that decay
geometrically with age -- the batch immediately before the target dominates the vote.

In [7]:
def path4_dynamic_fn(tr, ap, seeds=3, decay=0.5):
    batches = sorted(tr["batch"].unique())
    per_batch_probs = [baseline_fn(tr[tr["batch"] == b], ap, seeds=seeds) for b in batches]
    w = np.array([decay ** (len(batches) - 1 - i) for i in range(len(batches))])
    w = w / w.sum()
    return sum(wi * p for wi, p in zip(w, per_batch_probs))


print("=== path 4: dynamic (recency-weighted) ensemble ===")
P4_PROBS, P4_Y = eval_path(lambda tr, ap: path4_dynamic_fn(tr, ap, seeds=3))
ALL_PROBS["path4_dynamic"] = P4_PROBS
results.append(summarize("path4_dynamic", P4_PROBS, P4_Y))
pd.DataFrame(results)


=== path 4: dynamic (recency-weighted) ensemble ===
  [56s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path1_batchstd,0.795339,0.760929
2,path2_pca,0.880155,0.898716
3,path3_dann,0.760262,0.717368
4,path4_dynamic,0.854647,0.861079


## Path 5 -- semi-supervised pseudo-labeling

Train on the source batches, predict the target batch, keep predictions above `conf_thresh` as
pseudo-labels, retrain on source + pseudo-labeled target, predict again. True target labels are
never touched until the final `f1_of` scoring.

In [8]:
def path5_pseudolabel_fn(tr, ap, seeds=5, conf_thresh=0.9):
    (Xt, Xa), fscale = prep(tr, ap)
    y = np.searchsorted(CLASSES, tr["gas_class"].values)
    probs1 = train_seeds(Xt, y, Xa, fscale, n_seeds=seeds).mean(0)
    conf = probs1.max(1)
    mask = conf >= conf_thresh
    if mask.sum() == 0:
        return probs1
    Xt2 = np.vstack([Xt, Xa[mask]])
    y2 = np.concatenate([y, probs1[mask].argmax(1)])
    return train_seeds(Xt2, y2, Xa, fscale, n_seeds=seeds).mean(0)


print("=== path 5: pseudo-labeling self-training ===")
P5_PROBS, P5_Y = eval_path(lambda tr, ap: path5_pseudolabel_fn(tr, ap, seeds=5))
ALL_PROBS["path5_pseudolabel"] = P5_PROBS
results.append(summarize("path5_pseudolabel", P5_PROBS, P5_Y))
pd.DataFrame(results)


=== path 5: pseudo-labeling self-training ===
  [189s]


,name,mean,large_drift
0,baseline,0.849122,0.860706
1,path1_batchstd,0.795339,0.760929
2,path2_pca,0.880155,0.898716
3,path3_dann,0.760262,0.717368
4,path4_dynamic,0.854647,0.861079
5,path5_pseudolabel,0.838653,0.841068


## Path 6 -- regime switching (nearest-regime training window)

Every other pathway trains on *all* prior batches. This one instead asks "which recurring
sensor regime is the target batch actually in?" and trains only on that regime's batches --
batches 1-9 are clustered (k-means, k=3) on their standardized feature centroids (unlabeled,
so no leakage), giving `TRAIN_REGIME`. For a target batch, its own centroid is scored against
those cluster centers to pick its regime, training data is filtered down to same-regime source
batches, and the winning Path 2 (PCA-align) model is fit on just that subset. Drift here isn't
assumed to be monotonic -- if the target batch's sensors drift back toward an earlier regime
(a real phenomenon in long-running sensor arrays), this reuses that older data instead of diluting
the fit with the more recent, differently-drifted batches in between. Cold start (fewer than
`min_regime_batches` same-regime source batches) falls back to the full pooled fit.

In [ ]:
from sklearn.cluster import KMeans

N_REGIMES = 3
_batch_ids = sorted(train["batch"].unique())
_centroid_mat = np.vstack([_c[b] for b in _batch_ids])
_regime_km = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=0).fit(_centroid_mat)
TRAIN_REGIME = {b: int(l) for b, l in zip(_batch_ids, _regime_km.labels_)}
print(f"regime assignment (train batches): {TRAIN_REGIME}")


def regime_of_df(df):
    """Nearest train-regime for any batch's features (unlabeled, via the k-means centroids above)."""
    centroid = _s.transform(df[FEAT]).mean(0, keepdims=True)
    return int(_regime_km.predict(centroid)[0])


def path6_regime_fn(tr, ap, seeds=5, n_components=40, min_regime_batches=2):
    ap_regime = regime_of_df(ap)
    same_regime = [b for b in tr["batch"].unique() if TRAIN_REGIME[b] == ap_regime]
    tr_regime = tr[tr["batch"].isin(same_regime)]
    if tr_regime["batch"].nunique() < min_regime_batches:
        tr_regime = tr  # cold start: this regime barely seen yet, fall back to the full pool
    return path2_pca_fn(tr_regime, ap, seeds=seeds, n_components=n_components)


print("=== path 6: regime switching (same-regime training window + PCA-align) ===")
P6_PROBS, P6_Y = eval_path(lambda tr, ap: path6_regime_fn(tr, ap, seeds=5))
ALL_PROBS["path6_regime"] = P6_PROBS
results.append(summarize("path6_regime", P6_PROBS, P6_Y))
pd.DataFrame(results)

## Combine each pathway with the MLP baseline

Simple probability blend, `alpha * pathway + (1 - alpha) * baseline`, swept over a small grid.
Reuses the probabilities already computed above -- no retraining.

In [9]:
ALPHAS = [0.25, 0.5, 0.75, 1.0]
blend_results = []
for name in ["path1_batchstd", "path2_pca", "path3_dann", "path4_dynamic", "path5_pseudolabel", "path6_regime"]:
    probs, ys = ALL_PROBS[name], BASE_Y
    best = None
    for a in ALPHAS:
        blended = {vb: a * probs[vb] + (1 - a) * BASE_PROBS[vb] for vb in FOLDS}
        s = summarize(f"{name}+baseline (a={a})", blended, ys)
        if best is None or s["large_drift"] > best["large_drift"]:
            best = s
    blend_results.append(best)

summary_df = pd.DataFrame(results + blend_results).sort_values("large_drift", ascending=False)
print("=== full comparison, sorted by large-drift-fold macro-F1 ===")
summary_df


=== full comparison, sorted by large-drift-fold macro-F1 ===


,name,mean,large_drift
2,path2_pca,0.880155,0.898716
7,path2_pca+baseline (a=1.0),0.880155,0.898716
10,path5_pseudolabel+baseline (a=0.5),0.857744,0.868449
9,path4_dynamic+baseline (a=0.5),0.854839,0.867668
6,path1_batchstd+baseline (a=0.25),0.849973,0.863174
8,path3_dann+baseline (a=0.25),0.849214,0.861412
4,path4_dynamic,0.854647,0.861079
0,baseline,0.849122,0.860706
5,path5_pseudolabel,0.838653,0.841068
1,path1_batchstd,0.795339,0.760929


## Final: refit the best config on all 9 batches, predict batch 10, submit

Picks whichever row of `summary_df` scored highest on the large-drift folds -- baseline, a single
pathway, or a pathway+baseline blend -- and reuses that exact function (no retyped logic) to
produce the submission. Raise `seeds` here vs. the CV cells above now that it's one fit instead of
nine folds' worth.

In [10]:
FINAL_SEEDS = 15

candidates = {
    "baseline": lambda tr, ap: baseline_fn(tr, ap, seeds=FINAL_SEEDS),
    "path1_batchstd": lambda tr, ap: path1_batchstd_fn(tr, ap, seeds=FINAL_SEEDS),
    "path2_pca": lambda tr, ap: path2_pca_fn(tr, ap, seeds=FINAL_SEEDS),
    "path3_dann": lambda tr, ap: train_dann(tr, ap, seeds=FINAL_SEEDS),
    "path4_dynamic": lambda tr, ap: path4_dynamic_fn(tr, ap, seeds=5),
    "path5_pseudolabel": lambda tr, ap: path5_pseudolabel_fn(tr, ap, seeds=FINAL_SEEDS),
    "path6_regime": lambda tr, ap: path6_regime_fn(tr, ap, seeds=FINAL_SEEDS),
}
for row in blend_results:
    base_name = row["name"].split("+baseline")[0]
    alpha = float(row["name"].split("a=")[1].rstrip(")"))
    candidates[row["name"]] = (lambda base_name, alpha: lambda tr, ap: (
        alpha * candidates[base_name](tr, ap) + (1 - alpha) * baseline_fn(tr, ap, seeds=FINAL_SEEDS)
    ))(base_name, alpha)

best_name = summary_df.iloc[0]["name"]
print(f"best config by large-drift macro-F1: {best_name}")
P = candidates[best_name](train, test)

quota = np.full(K, TEST_QUOTA)
pred_free = np.array(CLASSES)[P.argmax(1)]
pred = np.array(CLASSES)[capped_greedy(sinkhorn(P, quota), quota)]
print(f"the balanced-assignment constraint changed {(pred != pred_free).mean():.1%} of the 3600 predictions")

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_pathways.csv", index=False)
print("wrote data/submission_pathways.csv")


best config by large-drift macro-F1: path2_pca
the balanced-assignment constraint changed 11.9% of the 3600 predictions
wrote data/submission_pathways.csv
